# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# The metadata object can be serialized to JSON for display
metadata_json = dataset.metadata.to_json()
print(f"\033[1m{metadata_json['name']}\033[0m: {metadata_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs. The `@id` field is used to reference all entities.


In [ ]:
# Print all record sets (tables) and their @id
metadata = dataset.metadata
record_sets = metadata.get('recordSet', [])
if not record_sets:
    print('No record sets found in the top-level metadata. Attempting to extract record sets from hasPart...')
    # Fallback: try from hasPart if available
    record_sets = []
    if 'hasPart' in metadata:  # Defensive, but not expected here
        for part in metadata['hasPart']:
            if isinstance(part, dict) and part.get('@type', '') == 'RecordSet':
                record_sets.append(part)

# If still not found, attempt to access "recordSet" from the loaded metadata as dict
if isinstance(record_sets, dict):
    record_sets = [record_sets]

if not record_sets:
    print('No record sets found.')
else:
    print('Record sets and their @id:')
    # In this dataset, recordSet may be empty, so try to obtain from metadata's 'distribution' or 'hasPart' if relevant
    for record_set in record_sets:
        rid = record_set['@id'] if isinstance(record_set, dict) and '@id' in record_set else record_set
        print(f'  @id: {rid}')

# For the FAIR^2 dataset, ids are often structured: Try listing record sets by inspecting the dataset:
all_record_set_ids = dataset.record_set_ids
print('Available record set ids:')
for rid in all_record_set_ids:
    print('  ', rid)

# Show some sample records and their keys/fields from a chosen record set
if all_record_set_ids:
    record_set_id = all_record_set_ids[0]
    print(f'\nSample records from record set {record_set_id}:')
    rows = list(dataset.records(record_set=record_set_id))
    if rows:
        print('Available fields (columns/@id):')
        for key in rows[0].keys():
            print('  ', key)
        print('\nFirst record:')
        print(rows[0])

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all available record sets
# All record_set_ids are referenced by their @id
record_sets = dataset.record_set_ids
dataframes = {}

for record_set_id in record_sets:
    print(f'Loading records for record set: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f'  Loaded {len(records)} records.')

# For demonstration, pick the first record set
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"\nColumns in main data table (@id={main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like outlier removal, transformations, or grouping data.


In [ ]:
# Select a numeric field for analysis
# Inspect columns for common numeric fields (e.g., 'coverage', 'mw', 'abundance', ...)
df = dataframes[main_record_set_id]
numeric_candidates = [col for col in df.columns if df[col].dtype != object]
if not numeric_candidates:
    # Try to coerce potential numeric columns to float
    possible_numerics = [col for col in df.columns if any(x in col.lower() for x in ['coverage', 'mw', 'abund', 'count', 'copy', 'percent', 'number', 'peptide', 'quant'])]
    for col in possible_numerics:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    numeric_candidates = [col for col in df.columns if df[col].dtype != object]

if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    raise ValueError('No numeric field could be detected for EDA. Consider inspecting DataFrame columns and types.')

threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {round(threshold, 2)} (mean):")
display(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if present
possible_group_fields = [col for col in df.columns if (df[col].dtype == object and len(df[col].unique()) < min(8, len(df)//4))]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    display(grouped_df)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='royalblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Boxplot of the normalized field if a grouping field is available
if group_field is not None:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field, y=f"{numeric_field}_normalized", data=filtered_df)
    plt.title(f'Normalized {numeric_field} for each group in {group_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- The FAIR² mass spectrometry dataset was loaded using `mlcroissant` and investigated interactively.
- All references to fields, record sets, and other entities used their `@id` as required by the Croissant spec and this notebook's guidelines.
- We identified key numeric and categorical fields for exploratory analysis, normalized values, and visualized their distributions.
- The notebook structure allows straightforward extension for more complex workflows, including downstream machine learning or domain-specific analysis.
